## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:

api_key = os.getenv('EMBEDDING_API_KEY')

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

False

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
self_MODEL = "qwen3.5-plus-2026-02-15"

In [5]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    temperature=0,
    model_name=self_MODEL,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=api_key,

)

### These LangChain objects implement the method `invoke()`

In [7]:
retriever.invoke("Who is Avery?")

[Document(id='ca41fce3-d5fd-4fc4-bbf0-9a3f6690c218', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [8]:
from week5.embedding import text_embedding
from week5.milvus_op import select_by_vector

question = "Who is Avery?"

question_vector = text_embedding(question)
res = select_by_vector(question_vector, 5)
print(res)

[{'similarity': 0.8754777908325195, 'text': '- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership was regained with innovative approaches to personalized insurance solutions. Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.'}, {'similarity': 0.9225050210952759, 'text': '# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk managem

In [ ]:
llm.invoke("Who is Avery?")

## Time to put this together!

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!